# Improvement round (Option A)

1. **scatter2d with more Gaussians.** The first run stopped at about 550 Gaussians, so two larger settings are tried here.
2. **scatter3d on real cells** (BBBC024 = 3D HL60 nuclei made with CytoPacq) instead of the synthetic ones.
3. **Comparison** of old vs new results.

`Runtime → Change runtime type → T4 GPU`, then run all cells (**Runtime → Run all**). Takes roughly 30–60 min.
New results go to `MyDrive/thesis/outputs/improved/`; the first run in `outputs/final/` is untouched.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/thesis/data"
OLD = "/content/drive/MyDrive/thesis/outputs/final"
NEW = "/content/drive/MyDrive/thesis/outputs/improved"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
%%writefile /content/thesis_gs.py
#!/usr/bin/env python3
"""
thesis_gs.py — Gaussian Splatting for physical inverse problems (InverseBench data)
==================================================================================

One script, three tasks:

  scatter2d : 2D Gaussians reconstruct real InverseBench cell images from scattered-field
              measurements (360/180/60 receivers). Baseline: plain pixel optimisation.
  scatter3d : 3D Gaussians reconstruct 3D cell volumes. Physics = the InverseBench 2D scattering
              operator applied slice by slice ("Option A"). Volumes come from CytoPacq files
              (--volumes DIR with .tif/.tiff/.npy) or, if none are given, from a built-in
              synthetic HL60-like nucleus generator. Baseline: plain voxel optimisation.
  ns4d      : 4D Gaussian Splatting (2D space + time) on Navier-Stokes vorticity.
              Canonical 2D Gaussians + a deformation network over time (4DGS design).
              Deformation = "mlp" (4DGS-style) or "mamba" (selective state-space model over time),
              trained from (optionally sparse) observations; evaluated on all frames incl.
              held-out time steps.

Each task has a "direct" mode (fit ground truth; sanity check) where it makes sense, and reports
PSNR / SSIM / relative L2 / #Gaussians / time. Everything is written to --out.

Colab usage (T4 GPU):
    from google.colab import drive; drive.mount('/content/drive')
    !python /content/drive/MyDrive/thesis/thesis_gs.py --task all \
        --data_dir /content/drive/MyDrive/thesis/data --out /content/drive/MyDrive/thesis/outputs/final

Quick smoke test (minutes, tiny settings):  add --quick
"""
import argparse
import glob
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================ setup helpers
CALTECH = "https://data.caltech.edu/records/zg89b-mpv16/files"
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


try:
    import lmdb
except ImportError:
    pip_install("lmdb")
    import lmdb


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def ensure_inversebench(ib_dir):
    if not os.path.isdir(os.path.join(ib_dir, "inverse_problems")):
        log("Cloning InverseBench ->", ib_dir)
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/devzhk/InverseBench.git", ib_dir], check=True)
    if ib_dir not in sys.path:
        sys.path.insert(0, ib_dir)


def ensure_lmdb_dataset(name, data_dir, local_dir):
    """Return a folder containing data.mdb for <name> (e.g. 'inv-scatter-test').
    Uses <data_dir>/<name>.zip if present, otherwise downloads it from CaltechDATA."""
    target = os.path.join(local_dir, name)
    if glob.glob(f"{target}/**/data.mdb", recursive=True):
        return os.path.dirname(glob.glob(f"{target}/**/data.mdb", recursive=True)[0])
    os.makedirs(data_dir, exist_ok=True)
    zip_path = os.path.join(data_dir, f"{name}.zip")
    if not os.path.exists(zip_path):
        import requests
        log(f"Downloading {name}.zip from CaltechDATA ...")
        r = requests.get(f"{CALTECH}/{name}.zip?download=1", headers=UA, timeout=300)
        if r.status_code != 200 or r.content[:2] != b"PK":
            raise RuntimeError(f"Download of {name}.zip failed (HTTP {r.status_code}). Download it manually from "
                               f"https://data.caltech.edu/records/zg89b-mpv16 and put it at {zip_path}")
        with open(zip_path, "wb") as f:
            f.write(r.content)
    shutil.rmtree(target, ignore_errors=True)
    zipfile.ZipFile(zip_path).extractall(target)
    hits = glob.glob(f"{target}/**/data.mdb", recursive=True)
    if not hits:
        raise RuntimeError(f"No data.mdb inside {zip_path}")
    return os.path.dirname(hits[0])


def read_lmdb_images(folder, n, res=128):
    env = lmdb.open(folder, readonly=True, lock=False, create=False)
    with env.begin() as txn:
        total = txn.stat()["entries"]
        out = [np.frombuffer(txn.get(str(i).encode()), dtype=np.float32).reshape(res, res).copy()
               for i in range(min(n, total))]
    env.close()
    return out


# ============================================================================ metrics
def psnr(x, gt, data_range=1.0):
    mse = torch.mean((x - gt) ** 2).item()
    return 10 * math.log10(data_range ** 2 / max(mse, 1e-12))


def ssim(x, gt, data_range=1.0):
    try:
        from skimage.metrics import structural_similarity
    except ImportError:
        pip_install("scikit-image")
        from skimage.metrics import structural_similarity
    return float(structural_similarity(x.detach().cpu().numpy().astype(np.float64),
                                       gt.detach().cpu().numpy().astype(np.float64), data_range=data_range))


def rel_l2(x, gt):
    return (torch.linalg.norm(x - gt) / torch.linalg.norm(gt).clamp(min=1e-12)).item()


# ============================================================================ Gaussian field (2D or 3D)
def quat_to_rot(q):
    q = F.normalize(q, dim=-1)
    w, x, y, z = q.unbind(-1)
    return torch.stack([
        1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y),
        2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x),
        2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)], -1).view(*q.shape[:-1], 3, 3)


class GaussianField:
    """N anisotropic Gaussians on a regular grid, one parameter tensor of shape (N, P).

    2D layout: [mu(2), log_s(2), theta(1), amp(1)]            P = 6
    3D layout: [mu(3), log_s(3), quat(4), amp(1)]             P = 11
    Coordinates are in grid-index units, in array axis order ((y,x) or (z,y,x)).
    amp is softplus(raw) >= 0 when nonneg=True (permittivity), raw signed otherwise (vorticity).
    Each Gaussian is evaluated only in a local window of +-h cells around its centre (the "3 sigma"
    trick from 3DGS), so cost is O(N * (2h+1)^dim), independent of the grid size.
    """

    def __init__(self, dim, grid, h, nonneg=True, wrap=False, device="cpu"):
        self.dim, self.grid, self.h, self.nonneg, self.wrap = dim, tuple(grid), h, nonneg, wrap
        self.P = 6 if dim == 2 else 11
        self.device = device
        r = torch.arange(-h, h + 1, device=device)
        self.offsets = torch.stack(torch.meshgrid(*([r] * dim), indexing="ij"), -1).reshape(-1, dim)  # (K, dim)
        self.s_min, self.s_max = 0.35, h / 3.0

    # ---- parameter helpers
    def make(self, mu, scale, amp):
        n = mu.shape[0]
        log_s = torch.full((n, self.dim), math.log(scale), device=mu.device)
        if self.dim == 2:
            rot = torch.zeros(n, 1, device=mu.device)
        else:
            rot = torch.zeros(n, 4, device=mu.device)
            rot[:, 0] = 1.0
        a = torch.log(torch.expm1(amp.clamp(min=1e-4))) if self.nonneg else amp
        return torch.cat([mu, log_s, rot, a[:, None]], 1)

    def amplitude(self, p):
        return F.softplus(p[:, -1]) if self.nonneg else p[:, -1]

    def init_from_field(self, field, n, scale=2.0, generator=None):
        """Sample centres proportional to |field| (+10% uniform)."""
        w = field.abs().flatten()
        w = w + 0.1 * w.mean().clamp(min=1e-6)
        idx = torch.multinomial(w / w.sum(), n, replacement=True, generator=generator)
        coords = torch.stack(torch.unravel_index(idx, self.grid), -1).float()
        coords = coords + torch.rand(coords.shape, generator=generator, device=coords.device) - 0.5
        amp = field.flatten()[idx] * 0.5
        if self.nonneg:
            amp = amp.abs().clamp(min=0.02)
        return self.make(coords, scale, amp)

    def init_uniform(self, n, scale=3.0, amp=0.05, generator=None):
        g = torch.tensor(self.grid, device=self.device, dtype=torch.float32)
        mu = torch.rand(n, self.dim, generator=generator, device=self.device) * (g - 1)
        return self.make(mu, scale, torch.full((n,), amp, device=self.device))

    # ---- rendering
    def inv_cov(self, p):
        d = self.dim
        s = torch.exp(p[:, d:2 * d]).clamp(self.s_min, self.s_max)
        if d == 2:
            th = p[:, 4]
            c, sn = torch.cos(th), torch.sin(th)
            R = torch.stack([c, -sn, sn, c], -1).view(-1, 2, 2)
        else:
            R = quat_to_rot(p[:, 6:10])
        return R @ torch.diag_embed(1.0 / s ** 2) @ R.transpose(1, 2)

    def render(self, p):
        d = self.dim
        mu = p[:, :d]
        base = torch.round(mu).detach().long()
        coords = base[:, None, :] + self.offsets[None]                       # (N, K, d)
        diff = coords.float() - mu[:, None, :]
        q = torch.einsum("nkd,nde,nke->nk", diff, self.inv_cov(p), diff)
        val = self.amplitude(p)[:, None] * torch.exp(-0.5 * q)
        g = torch.tensor(self.grid, device=p.device)
        if self.wrap:
            coords = coords % g
        else:
            inside = ((coords >= 0) & (coords < g)).all(-1)
            val = val * inside
            coords = torch.minimum(coords.clamp(min=0), g - 1)
        strides = torch.tensor([int(np.prod(self.grid[i + 1:])) for i in range(d)], device=p.device)
        flat = (coords * strides).sum(-1)
        out = torch.zeros(int(np.prod(self.grid)), device=p.device, dtype=val.dtype)
        out = out.index_add(0, flat.flatten(), val.flatten())
        return out.view(self.grid)


# ============================================================================ scattering physics
class SliceScatter:
    """InverseBench linear inverse-scattering operator, applied to every 128x128 slice of a stack.
        y[z, t, r] = dx*dy * sum_p f[z, p] * conj(uinc[p, t]) * G[p, r]
    Identical to InverseScatter.forward(f, unnormalize=False) per slice (checked at start-up)."""

    def __init__(self, receivers, device):
        from inverse_problems.inverse_scatter import InverseScatter
        self.ib = InverseScatter(Lx=0.18, Ly=0.18, Nx=128, Ny=128, wave=6, numRec=receivers, numTrans=20,
                                 sensorRadius=1.6, sigma_noise=1e-4, unnorm_shift=1.0, unnorm_scale=0.5,
                                 svd=False, device=device)
        self.dxdy = self.ib.dx * self.ib.dy
        self.uc = torch.conj(self.ib.uinc_dom_set[..., 0].reshape(-1, 20)).to(device, torch.complex64)   # (P, T)
        self.G = self.ib.sensor_greens_function_set[..., 0].reshape(-1, receivers).to(device, torch.complex64)  # (P, R)

    def __call__(self, f):                       # f: (..., 128, 128) real -> (Z, T, R) complex
        f = f.reshape(-1, 128 * 128).to(torch.complex64)
        src = f[:, :, None] * self.uc[None]                      # (Z, P, T)
        return self.dxdy * torch.einsum("zpt,pr->ztr", src, self.G)

    def adjoint_real(self, r):                   # r: (Z, T, R) -> (Z, 128, 128) back-projection
        bp = torch.einsum("pr,ztr->zpt", self.G.conj(), r.to(torch.complex64)) * self.uc.conj()[None]
        return (self.dxdy * bp.sum(-1)).real.view(-1, 128, 128)

    def check(self, f2d):
        y_ib = self.ib.forward(f2d[None, None].double(), unnormalize=False)[0]
        return ((self(f2d)[0] - y_ib.to(torch.complex64)).abs().max() / y_ib.abs().max()).item()


def add_noise(y, sigma, gen):
    n = torch.randn(y.shape, generator=gen, device=y.device, dtype=torch.float32) \
        + 1j * torch.randn(y.shape, generator=gen, device=y.device, dtype=torch.float32)
    return y + sigma * n / math.sqrt(2)


# ============================================================================ 3D cell volumes
def synthetic_nuclei_volume(D, H, W, seed):
    """Stand-in for CytoPacq HL60 nuclei: 1-6 soft rotated ellipsoids with smooth internal texture, 0..1."""
    rng = np.random.default_rng(seed)
    z, y, x = np.meshgrid(np.arange(D), np.arange(H), np.arange(W), indexing="ij")
    vol = np.zeros((D, H, W), np.float32)
    for _ in range(rng.integers(1, 7)):
        c = np.array([rng.uniform(0.35, 0.65) * D, rng.uniform(0.25, 0.75) * H, rng.uniform(0.25, 0.75) * W])
        r = np.array([rng.uniform(0.15, 0.25) * D, rng.uniform(8, 16), rng.uniform(8, 16)])
        a = rng.uniform(0, np.pi)
        yy, xx = y - c[1], x - c[2]
        yr, xr = np.cos(a) * yy - np.sin(a) * xx, np.sin(a) * yy + np.cos(a) * xx
        dist = np.sqrt(((z - c[0]) / r[0]) ** 2 + (yr / r[1]) ** 2 + (xr / r[2]) ** 2)
        shape = 1 / (1 + np.exp((dist - 1) * 12))
        k = rng.normal(size=(3, 3))
        tex = 1 - 0.25 * (np.sin(k[0, 0] * z / 3 + k[0, 1] * y / 5 + k[0, 2] * x / 5) *
                          np.cos(k[1, 0] * z / 4 + k[1, 1] * y / 6 + k[1, 2] * x / 6)) ** 2
        vol = np.maximum(vol, shape * tex * rng.uniform(0.6, 1.0))
    return np.clip(vol, 0, 1).astype(np.float32)


def load_volumes(folder, n, D, device, crop=0.5):
    """Load 3D volumes (.tif/.tiff/.npy, shape Z x Y x X), e.g. BBBC024 / CytoPacq ground-truth masks.
    Label masks (one integer per nucleus) become a 0/1 foreground mask; the central `crop` fraction in
    Y and X is kept (so a few nuclei fill the field, as in InverseBench); resized to D x 128 x 128, 0..1."""
    files = sorted(sum([glob.glob(os.path.join(folder, "**", e), recursive=True)
                        for e in ("*.tif", "*.tiff", "*.npy")], []))[:n]
    if not files:
        raise RuntimeError(f"No .tif/.tiff/.npy volumes in {folder}")
    vols = []
    for f in files:
        if f.endswith(".npy"):
            v = np.load(f)
        else:
            try:
                import tifffile
            except ImportError:
                pip_install("tifffile")
                import tifffile
            v = tifffile.imread(f)
        v = np.asarray(v).squeeze()
        assert v.ndim == 3, f"{f}: expected a 3D volume, got shape {v.shape}"
        if np.issubdtype(v.dtype, np.integer) and len(np.unique(v)) <= 256:
            v = v > 0                                         # label mask -> foreground
        v = torch.from_numpy(v.astype(np.float32))
        if crop < 1:
            _, Y, X = v.shape
            cy, cx = max(1, int(Y * crop)), max(1, int(X * crop))
            v = v[:, (Y - cy) // 2:(Y - cy) // 2 + cy, (X - cx) // 2:(X - cx) // 2 + cx]
        v = F.interpolate(v[None, None], size=(D, 128, 128), mode="trilinear", align_corners=False)[0, 0]
        v = (v - v.min()) / (v.max() - v.min()).clamp(min=1e-8)
        vols.append(v.to(device))
        log(f"  loaded {os.path.basename(f)} -> {tuple(v.shape)}, foreground {float((v > 0.5).float().mean()):.1%}")
    return vols


# ============================================================================ scattering fit (2D and 3D)
def cosine_lr(opt, base, it, total):
    for pg in opt.param_groups:
        pg["lr"] = pg.get("base_lr", base) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * it / total)))


def fit_scatter(gt, field, mode, op=None, y=None, n_init=400, n_max=4000, iters=3000, lr=0.05,
                densify_every=250, n_add=None, densify_until=0.6, seed=0, log_every=500):
    """mode='direct': MSE to ground truth.  mode='physics': ||A f - y||^2/||y||^2 (gt only for metrics)."""
    g = torch.Generator(device=gt.device).manual_seed(seed)
    n_add = n_add or max(50, n_init // 4)
    p = (field.init_from_field(gt, n_init, generator=g) if mode == "direct"
         else field.init_uniform(n_init, generator=g)).requires_grad_(True)
    opt = torch.optim.Adam([p], lr=lr)
    ynorm = (y.abs() ** 2).sum() if y is not None else None
    t0 = time.time()
    for it in range(1, iters + 1):
        img = field.render(p)
        loss = F.mse_loss(img, gt) if mode == "direct" else ((op(img) - y).abs() ** 2).sum() / ynorm
        opt.zero_grad()
        loss.backward()
        opt.step()
        cosine_lr(opt, lr, it, iters)
        if densify_every and it % densify_every == 0 and it <= densify_until * iters:
            with torch.no_grad():
                img = field.render(p)
                if mode == "direct":
                    res = gt - img
                else:
                    res = op.adjoint_real(y - op(img)).view(gt.shape)
                    res = res / res.abs().max().clamp(min=1e-12) * 0.3
                keep = field.amplitude(p) > 1e-3
                k = max(0, min(n_add, n_max - int(keep.sum())))
                new = []
                if k:
                    idx = torch.topk(res.abs().flatten(), k).indices
                    mu = torch.stack(torch.unravel_index(idx, field.grid), -1).float()
                    new = [field.make(mu, 1.5, res.flatten()[idx].abs().clamp(min=0.02))]
                p = torch.cat([p.detach()[keep]] + new, 0).requires_grad_(True)
                cur = opt.param_groups[0]["lr"]
                opt = torch.optim.Adam([p], lr=cur)
                opt.param_groups[0]["base_lr"] = lr
        if it % log_every == 0 or it == iters:
            with torch.no_grad():
                log(f"    iter {it:5d}  loss {loss.item():.3e}  PSNR {psnr(field.render(p).clamp(0, 1), gt):6.2f} dB  #G {p.shape[0]}")
    with torch.no_grad():
        rec = field.render(p).clamp(0, 1)
    return p.detach(), rec, {"psnr": psnr(rec, gt), "ssim": ssim(rec, gt), "rel_l2": rel_l2(rec, gt),
                             "n_gauss": p.shape[0], "time_s": round(time.time() - t0, 1)}


def fit_grid_baseline(gt, op, y, iters=3000, lr=0.01):
    """Baseline: optimise every pixel/voxel directly with the same physics loss, projected to f >= 0."""
    x = torch.zeros_like(gt, requires_grad=True)
    opt = torch.optim.Adam([x], lr=lr)
    ynorm = (y.abs() ** 2).sum()
    t0 = time.time()
    for _ in range(iters):
        loss = ((op(x) - y).abs() ** 2).sum() / ynorm
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            x.clamp_(0, 1)
    rec = x.detach()
    return rec, {"psnr": psnr(rec, gt), "ssim": ssim(rec, gt), "rel_l2": rel_l2(rec, gt),
                 "n_gauss": 0, "time_s": round(time.time() - t0, 1)}


def run_scatter(args, dim):
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    task = f"scatter{dim}d"
    out = os.path.join(args.out, task)
    os.makedirs(out, exist_ok=True)
    dev = args.device
    ensure_inversebench(args.ib_dir)

    # ---- data
    if dim == 2:
        folder = ensure_lmdb_dataset("inv-scatter-test", args.data_dir, args.local_dir)
        gts = [torch.from_numpy(im).to(dev) for im in read_lmdb_images(folder, args.samples)]
        field = GaussianField(2, (128, 128), h=args.window, nonneg=True, device=dev)
        source = "InverseBench inv-scatter-test"
    else:
        if args.volumes:
            gts = load_volumes(args.volumes, args.samples, args.depth, dev, crop=args.crop)
            source = f"CytoPacq volumes from {args.volumes}"
        else:
            gts = [torch.from_numpy(synthetic_nuclei_volume(args.depth, 128, 128, seed=100 + i)).to(dev)
                   for i in range(args.samples)]
            source = "synthetic HL60-like nuclei (no --volumes given)"
        field = GaussianField(3, (args.depth, 128, 128), h=args.window, nonneg=True, device=dev)
    log(f"[{task}] {len(gts)} samples, shape {tuple(gts[0].shape)}, source: {source}")

    n_init = args.n_init or args.n_gauss // (8 if dim == 2 else 4)
    dens = dict(n_add=args.densify_add, densify_until=args.densify_until)
    rows, examples = [], {}

    # ---- A: direct fit (representation check)
    if not args.skip_direct:
        for i, gt in enumerate(gts):
            log(f"[{task}] A direct fit, sample {i}")
            p, rec, r = fit_scatter(gt, field, "direct", n_init=n_init, n_max=args.n_gauss, iters=args.iters,
                                    seed=i, log_every=max(args.iters // 4, 1))
            rows.append({"experiment": "A_direct", "receivers": None, "sample": i, "method": "gaussians", **r})
            if i == 0:
                examples["direct"] = rec.cpu()
                torch.save(p.cpu(), os.path.join(out, "A_direct_sample0_params.pt"))

    # ---- B: reconstruction from measurements
    for R in args.receivers:
        op = SliceScatter(R, dev)
        log(f"[{task}] B physics, {R} receivers (operator check vs InverseBench: {op.check(gts[0].view(-1, 128, 128)[0]):.1e})")
        for i, gt in enumerate(gts):
            gen = torch.Generator(device=dev).manual_seed(1000 + i)
            y = add_noise(op(gt), args.noise, gen)
            log(f"[{task}]   sample {i}: Gaussians")
            p, rec_g, rg = fit_scatter(gt, field, "physics", op=op, y=y, n_init=n_init, n_max=args.n_gauss,
                                       iters=args.iters, seed=i, log_every=max(args.iters // 4, 1), **dens)
            log(f"[{task}]   sample {i}: grid baseline")
            rec_b, rb = fit_grid_baseline(gt, op, y, iters=args.iters)
            log(f"[{task}]   -> Gaussians {rg['psnr']:.2f} dB / SSIM {rg['ssim']:.3f}   "
                f"grid {rb['psnr']:.2f} dB / SSIM {rb['ssim']:.3f}")
            rows += [{"experiment": "B_physics", "receivers": R, "sample": i, "method": "gaussians", **rg},
                     {"experiment": "B_physics", "receivers": R, "sample": i, "method": "grid_baseline", **rb}]
            if i == 0:
                examples[R] = (rec_g.cpu(), rec_b.cpu(), rg, rb)
                torch.save(p.cpu(), os.path.join(out, f"B_physics_R{R}_sample0_params.pt"))

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "results.csv"), index=False)
    summ = df.groupby(["experiment", "receivers", "method"], dropna=False)[
        ["psnr", "ssim", "rel_l2", "n_gauss", "time_s"]].mean().round(3)
    summ.to_csv(os.path.join(out, "summary.csv"))
    log(f"[{task}] summary\n{summ.to_string()}")

    # ---- figure: sample 0 (middle slice for 3D)
    def sl(v):
        return v[v.shape[0] // 2] if v.ndim == 3 else v
    cols = [("ground truth", sl(gts[0].cpu()))]
    if "direct" in examples:
        cols.append(("direct fit", sl(examples["direct"])))
    for R in args.receivers:
        g, b, rg, rb = examples[R]
        cols += [(f"Gaussians {R} rec.\n{rg['psnr']:.1f} dB", sl(g)), (f"grid {R} rec.\n{rb['psnr']:.1f} dB", sl(b))]
    fig, ax = plt.subplots(1, len(cols), figsize=(2.6 * len(cols), 3))
    for a, (t, im) in zip(np.atleast_1d(ax), cols):
        a.imshow(im.numpy(), cmap="gray", vmin=0, vmax=1)
        a.set_title(t, fontsize=8)
        a.axis("off")
    fig.suptitle(f"{task}: sample 0" + (" (middle slice)" if dim == 3 else ""))
    fig.tight_layout()
    fig.savefig(os.path.join(out, "reconstructions.png"), dpi=130)
    plt.close(fig)
    return summ


# ============================================================================ Navier-Stokes data
def ns_sequences(args, dev):
    """Initial vorticity from InverseBench NS test set, rolled out with InverseBench's solver.
    Returns list of tensors (T, 128, 128), normalised by 10 (vorticity range is about +-10)."""
    cache = os.path.join(args.out, "ns4d", f"sequences_n{args.samples}_T{args.frames}_dt{args.frame_dt}.pt")
    if os.path.exists(cache):
        log(f"[ns4d] loading cached sequences {cache}")
        return [s.to(dev) for s in torch.load(cache)]
    from inverse_problems.navier_stokes import NavierStokes2d
    folder = ensure_lmdb_dataset("navier-stokes-test", args.data_dir, args.local_dir)
    w0s = read_lmdb_images(folder, args.samples)
    solver = NavierStokes2d(128, 128, 2 * math.pi, 2 * math.pi, device=dev, dtype=torch.float64)
    t = torch.linspace(0, 2 * math.pi, 129, device=dev, dtype=torch.float64)[:-1]
    _, yy = torch.meshgrid(t, t, indexing="ij")
    force = -4 * torch.cos(4.0 * yy)
    seqs = []
    for i, w0 in enumerate(w0s):
        t0 = time.time()
        w = torch.from_numpy(w0).to(dev, torch.float64)[None]
        frames = [w[0].float()]
        for _ in range(args.frames - 1):
            w = solver.solve(w, force, args.frame_dt, 200.0, adaptive=True)
            frames.append(w[0].float())
        seqs.append(torch.stack(frames) / 10.0)
        log(f"[ns4d] rolled out sample {i}: {args.frames} frames x dt={args.frame_dt} in {time.time() - t0:.1f}s")
    torch.save([s.cpu() for s in seqs], cache)
    return seqs


# ============================================================================ deformation networks
def fourier(x, n_freq):
    """x in [0,1] -> [sin(2^k pi x), cos(2^k pi x)]_k  (periodic-friendly)."""
    f = (2.0 ** torch.arange(n_freq, device=x.device)) * math.pi
    a = x[..., None] * f
    return torch.cat([torch.sin(a), torch.cos(a)], -1).flatten(-2)


class DeformMLP(nn.Module):
    """4DGS-style: (canonical Gaussian position, time) -> parameter offsets, independently per (Gaussian, t)."""

    def __init__(self, n_out=6, n_freq=6, width=128, depth=3):
        super().__init__()
        self.n_freq = n_freq
        layers, d_in = [], 3 * 2 * n_freq
        for _ in range(depth):
            layers += [nn.Linear(d_in, width), nn.SiLU()]
            d_in = width
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, n_out)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, mu_norm, t_all, idx):
        """mu_norm (N,2) in [0,1], t_all (T,) in [0,1], idx: frame indices -> (len(idx), N, n_out)."""
        t = t_all[idx]
        N, B = mu_norm.shape[0], t.shape[0]
        x = torch.cat([fourier(mu_norm, self.n_freq)[None].expand(B, N, -1),
                       fourier(t[:, None, None].expand(B, N, 1), self.n_freq)], -1)
        return self.head(self.body(x))


class MambaBlock(nn.Module):
    """Minimal pure-PyTorch Mamba (selective state-space) block, Gu & Dao 2023. Causal over the sequence."""

    def __init__(self, d_model, d_state=16, expand=2, d_conv=4):
        super().__init__()
        di = expand * d_model
        self.di, self.ds = di, d_state
        self.dt_rank = max(1, math.ceil(d_model / 16))
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * di)
        self.conv = nn.Conv1d(di, di, d_conv, groups=di, padding=d_conv - 1)
        self.x_proj = nn.Linear(di, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, di)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)).repeat(di, 1))
        self.D = nn.Parameter(torch.ones(di))
        self.out_proj = nn.Linear(di, d_model)

    def forward(self, x):                                     # x: (B, L, d)
        B, L, _ = x.shape
        xz = self.in_proj(self.norm(x))
        u, z = xz.chunk(2, -1)
        u = F.silu(self.conv(u.transpose(1, 2))[..., :L].transpose(1, 2))
        dt, Bm, Cm = self.x_proj(u).split([self.dt_rank, self.ds, self.ds], -1)
        dt = F.softplus(self.dt_proj(dt))                     # (B, L, di)
        A = -torch.exp(self.A_log)                            # (di, ds)
        h = torch.zeros(B, self.di, self.ds, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):                                    # selective scan
            h = torch.exp(dt[:, t, :, None] * A) * h + dt[:, t, :, None] * Bm[:, t, None, :] * u[:, t, :, None]
            ys.append((h * Cm[:, t, None, :]).sum(-1))
        y = (torch.stack(ys, 1) + self.D * u) * F.silu(z)
        return x + self.out_proj(y)


class DeformMamba(nn.Module):
    """Deformation as a sequence model over time: each Gaussian's trajectory is one sequence of length T,
    processed by Mamba blocks, so the offset at frame t depends on the Gaussian's state at earlier frames."""

    def __init__(self, n_out=6, n_freq=6, d_model=64, n_layers=2):
        super().__init__()
        self.n_freq = n_freq
        self.inp = nn.Linear(3 * 2 * n_freq, d_model)
        self.blocks = nn.ModuleList([MambaBlock(d_model) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_out)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, mu_norm, t_all, idx):
        N, T = mu_norm.shape[0], t_all.shape[0]
        x = torch.cat([fourier(mu_norm, self.n_freq)[:, None].expand(N, T, -1),
                       fourier(t_all[None, :, None].expand(N, T, 1), self.n_freq)], -1)
        h = self.inp(x)
        for b in self.blocks:
            h = b(h)
        out = self.head(self.norm(h))                         # (N, T, n_out)
        return out[:, idx].transpose(0, 1)                    # (len(idx), N, n_out)


class StaticDeform(nn.Module):
    """No deformation (ablation): the same Gaussians for every frame."""

    def forward(self, mu_norm, t_all, idx):
        return torch.zeros(len(idx), mu_norm.shape[0], 6, device=mu_norm.device)


def apply_deform(canon, delta, move_scale=8.0):
    """canon (N,6) + delta (B,N,6) -> per-frame params (B,N,6). Positions move in pixels."""
    d = delta.clone()
    d[..., :2] = d[..., :2] * move_scale
    return canon[None] + d


# ============================================================================ 4D training
def run_ns4d(args):
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    out = os.path.join(args.out, "ns4d")
    os.makedirs(out, exist_ok=True)
    dev = args.device
    ensure_inversebench(args.ib_dir)
    seqs = ns_sequences(args, dev)
    T = seqs[0].shape[0]
    t_all = torch.linspace(0, 1, T, device=dev)
    test_idx = torch.arange(2, T, 4, device=dev)                       # held-out time steps
    train_idx = torch.tensor([k for k in range(T) if k not in set(test_idx.tolist())], device=dev)
    field = GaussianField(2, (128, 128), h=args.window, nonneg=False, wrap=True, device=dev)  # periodic domain
    log(f"[ns4d] {len(seqs)} sequences x {T} frames; train frames {len(train_idx)}, held-out {len(test_idx)}; "
        f"observed pixel ratios {args.obs_ratios}; deformations {args.deform}")

    rows, examples = [], {}
    for ratio in args.obs_ratios:
        for si, seq in enumerate(seqs):
            gen = torch.Generator(device=dev).manual_seed(500 + si)
            mask = (torch.rand(seq.shape, generator=gen, device=dev) < ratio).float()   # observed pixels per frame
            obs = seq + args.ns_noise * torch.randn(seq.shape, generator=gen, device=dev)
            for kind in args.deform:
                torch.manual_seed(si)
                net = {"mlp": DeformMLP, "mamba": DeformMamba, "static": StaticDeform}[kind]().to(dev)
                g0 = torch.Generator(device=dev).manual_seed(si)
                canon = field.init_from_field(obs[0] * mask[0], args.n_gauss, scale=2.0, generator=g0)
                canon.requires_grad_(True)
                groups = [{"params": [canon], "lr": args.lr_gauss, "base_lr": args.lr_gauss}]
                if any(True for _ in net.parameters()):
                    groups.append({"params": list(net.parameters()), "lr": args.lr_net, "base_lr": args.lr_net})
                opt = torch.optim.Adam(groups)
                t0 = time.time()
                log(f"[ns4d] ratio {ratio}, sample {si}, deform={kind}")
                for it in range(1, args.iters + 1):
                    bidx = train_idx[torch.randint(len(train_idx), (min(args.batch_frames, len(train_idx)),),
                                                   device=dev)]
                    params = apply_deform(canon, net(canon[:, :2] / 128.0, t_all, bidx))
                    pred = torch.stack([field.render(params[b]) for b in range(len(bidx))])
                    m = mask[bidx]
                    loss = (((pred - obs[bidx]) ** 2) * m).sum() / m.sum().clamp(min=1)
                    opt.zero_grad()
                    loss.backward()
                    opt.step()
                    cosine_lr(opt, None, it, args.iters)
                    if it % max(args.iters // 4, 1) == 0 or it == args.iters:
                        log(f"    iter {it:5d}  loss {loss.item():.3e}")
                train_time = time.time() - t0
                with torch.no_grad():
                    allp = apply_deform(canon, net(canon[:, :2] / 128.0, t_all, torch.arange(T, device=dev)))
                    rec = torch.stack([field.render(allp[k]) for k in range(T)])
                res = {}
                for name, idx in [("train_frames", train_idx), ("heldout_frames", test_idx)]:
                    res[f"{name}_rel_l2"] = float(np.mean([rel_l2(rec[k], seq[k]) for k in idx.tolist()]))
                    res[f"{name}_psnr"] = float(np.mean([psnr(rec[k], seq[k], 2.0) for k in idx.tolist()]))
                    res[f"{name}_ssim"] = float(np.mean([ssim(rec[k], seq[k], 2.0) for k in idx.tolist()]))
                n_net = sum(p.numel() for p in net.parameters())
                rows.append({"obs_ratio": ratio, "sample": si, "deform": kind, **res, "n_gauss": canon.shape[0],
                             "net_params": n_net, "time_s": round(train_time, 1)})
                log(f"    -> rel L2 train {res['train_frames_rel_l2']:.4f} | held-out {res['heldout_frames_rel_l2']:.4f}"
                    f"   PSNR held-out {res['heldout_frames_psnr']:.2f} dB  ({train_time:.0f}s)")
                if si == 0:
                    examples[(ratio, kind)] = rec.cpu()
                    torch.save({"canonical": canon.detach().cpu(), "net": net.state_dict()},
                               os.path.join(out, f"model_ratio{ratio}_{kind}_sample0.pt"))

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "results.csv"), index=False)
    cols = [c for c in df.columns if c.endswith(("rel_l2", "psnr", "ssim"))] + ["time_s"]
    summ = df.groupby(["obs_ratio", "deform"])[cols].mean().round(4)
    summ.to_csv(os.path.join(out, "summary.csv"))
    log(f"[ns4d] summary\n{summ.to_string()}")

    # figure: sample 0, a few frames incl. held-out ones, for every (ratio, deform)
    show = sorted(set([0, int(test_idx[0]), T // 2, int(test_idx[-1]), T - 1]))
    keys = list(examples.keys())
    fig, ax = plt.subplots(1 + len(keys), len(show), figsize=(2.2 * len(show), 2.2 * (1 + len(keys))), squeeze=False)
    for j, k in enumerate(show):
        tag = " (held-out)" if k in set(test_idx.tolist()) else ""
        ax[0, j].imshow(seqs[0][k].cpu(), cmap="RdBu_r", vmin=-1, vmax=1)
        ax[0, j].set_title(f"GT t={k}{tag}", fontsize=7)
        for i, key in enumerate(keys):
            ax[i + 1, j].imshow(examples[key][k], cmap="RdBu_r", vmin=-1, vmax=1)
            ax[i + 1, j].set_title(f"{key[1]}, obs {key[0]:.0%}", fontsize=7)
    for a in ax.flat:
        a.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "reconstructions.png"), dpi=130)
    plt.close(fig)
    return summ


# ============================================================================ main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--task", default="all", choices=["scatter2d", "scatter3d", "ns4d", "all"])
    ap.add_argument("--out", default="outputs")
    ap.add_argument("--data_dir", default="data", help="where the CaltechDATA zips are (or will be downloaded)")
    ap.add_argument("--local_dir", default="/tmp/thesis_data", help="fast local folder for unzipped data")
    ap.add_argument("--ib_dir", default="/tmp/InverseBench", help="InverseBench clone (cloned if missing)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--samples", type=int, default=3)
    ap.add_argument("--iters", type=int, default=3000)
    ap.add_argument("--n_gauss", type=int, default=None, help="max Gaussians (default 2000 2D / 8000 3D / 2000 4D)")
    ap.add_argument("--window", type=int, default=None, help="local window half-size (default 8 / 6 / 10)")
    # scattering
    ap.add_argument("--receivers", type=int, nargs="+", default=[360, 180, 60])
    ap.add_argument("--noise", type=float, default=1e-4, help="measurement noise (benchmark: 1e-4)")
    ap.add_argument("--volumes", default=None, help="folder with CytoPacq .tif/.npy volumes for scatter3d")
    ap.add_argument("--depth", type=int, default=32, help="number of z-slices for 3D volumes")
    ap.add_argument("--skip_direct", action="store_true")
    ap.add_argument("--crop", type=float, default=0.5, help="central Y/X fraction kept from loaded volumes")
    ap.add_argument("--n_init", type=int, default=None, help="initial Gaussians (default n_gauss/8 2D, /4 3D)")
    ap.add_argument("--densify_add", type=int, default=None, help="Gaussians added per densify step (default n_init/4)")
    ap.add_argument("--densify_until", type=float, default=0.6, help="densify during this fraction of iterations")
    # Navier-Stokes 4D
    ap.add_argument("--frames", type=int, default=21)
    ap.add_argument("--frame_dt", type=float, default=0.05)
    ap.add_argument("--obs_ratios", type=float, nargs="+", default=[1.0, 0.1])
    ap.add_argument("--ns_noise", type=float, default=0.0)
    ap.add_argument("--deform", nargs="+", default=["mlp", "mamba"], choices=["mlp", "mamba", "static"])
    ap.add_argument("--batch_frames", type=int, default=4)
    ap.add_argument("--lr_gauss", type=float, default=0.01)
    ap.add_argument("--lr_net", type=float, default=1e-3)
    ap.add_argument("--quick", action="store_true", help="tiny settings for a smoke test")
    args = ap.parse_args()

    if args.quick:
        args.samples, args.iters = 1, 40
        args.receivers, args.depth, args.frames = [360, 60], 16, 6
        args.obs_ratios = [0.1]
    os.makedirs(args.out, exist_ok=True)
    with open(os.path.join(args.out, "run_config.json"), "w") as f:
        json.dump(vars(args), f, indent=1)
    log(f"device={args.device}  task={args.task}  out={args.out}")
    if args.device == "cpu" and not args.quick:
        log("WARNING: running on CPU will be very slow. In Colab: Runtime -> Change runtime type -> T4 GPU.")

    tasks = ["scatter2d", "scatter3d", "ns4d"] if args.task == "all" else [args.task]
    user_n, user_w = args.n_gauss, args.window
    for task in tasks:
        defaults = {"scatter2d": (2000, 8), "scatter3d": (8000, 6), "ns4d": (2000, 10)}[task]
        args.n_gauss = user_n or (defaults[0] // (10 if args.quick else 1))
        args.window = user_w or defaults[1]
        t0 = time.time()
        if task == "scatter2d":
            run_scatter(args, 2)
        elif task == "scatter3d":
            run_scatter(args, 3)
        else:
            run_ns4d(args)
        log(f"===== {task} finished in {(time.time() - t0) / 60:.1f} min. Results in {os.path.join(args.out, task)}")


if __name__ == "__main__":
    main()

## 2. scatter2d with more Gaussians (two settings)

In [ ]:
# "more": start with 500 Gaussians, add 300 per densify step, keep densifying until 80% of training (max 4000)
!python /content/thesis_gs.py --task scatter2d --skip_direct --n_gauss 4000 --n_init 500 --densify_add 300 --densify_until 0.8 \
    --data_dir {DATA_DIR} --out {NEW}/more

In [ ]:
# "most": start with 1000 Gaussians, same densification
!python /content/thesis_gs.py --task scatter2d --skip_direct --n_gauss 4000 --n_init 1000 --densify_add 300 --densify_until 0.8 \
    --data_dir {DATA_DIR} --out {NEW}/most

## 3. scatter3d on real CytoPacq cells (BBBC024)

In [ ]:
import os, subprocess, zipfile, glob
VOL = "/content/drive/MyDrive/thesis/cytopacq"
os.makedirs(VOL, exist_ok=True)
name = "BBBC024_v1_c00_highSNR_foreground.zip"     # ground-truth 3D masks, 20 HL60 nuclei per volume
zip_path = f"{VOL}/{name}"
if not os.path.exists(zip_path):
    r = subprocess.run(["wget", "-q", "-O", zip_path, f"https://data.broadinstitute.org/bbbc/BBBC024/{name}"])
    if r.returncode != 0 or os.path.getsize(zip_path) < 1000:
        os.remove(zip_path)
        raise RuntimeError(f"Download failed. Get {name} from https://data.broadinstitute.org/bbbc/BBBC024/ "
                           f"in your browser, upload it to MyDrive/thesis/cytopacq/, and rerun this cell.")
if not glob.glob(f"{VOL}/**/*.tif*", recursive=True):
    zipfile.ZipFile(zip_path).extractall(VOL)
files = sorted(glob.glob(f"{VOL}/**/*.tif*", recursive=True))
print(len(files), "volumes, e.g.", [os.path.basename(f) for f in files[:3]])

In [ ]:
!python /content/thesis_gs.py --task scatter3d --volumes {VOL} --crop 0.5 \
    --data_dir {DATA_DIR} --out {NEW}/bbbc024

## 4. Compare old vs new

In [ ]:
import os
import pandas as pd
from IPython.display import Image, display
cols = ["experiment", "receivers", "method", "psnr", "ssim", "n_gauss"]
runs = {"scatter2d  first run": f"{OLD}/scatter2d", "scatter2d  more": f"{NEW}/more/scatter2d",
        "scatter2d  most": f"{NEW}/most/scatter2d", "scatter3d  synthetic (first run)": f"{OLD}/scatter3d",
        "scatter3d  BBBC024 real cells": f"{NEW}/bbbc024/scatter3d"}
for label, folder in runs.items():
    if os.path.exists(f"{folder}/summary.csv"):
        print(f"\n===== {label} =====")
        display(pd.read_csv(f"{folder}/summary.csv")[cols])
for folder in [f"{NEW}/more/scatter2d", f"{NEW}/most/scatter2d", f"{NEW}/bbbc024/scatter3d"]:
    if os.path.exists(f"{folder}/reconstructions.png"):
        print(folder); display(Image(filename=f"{folder}/reconstructions.png"))